In [ ]:
vol = 3

## Parse EPUB

In [ ]:
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

def extract_paragraphs_from_epub(epub_path):
    book = epub.read_epub(epub_path)
    chapters = []
    
    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        soup = BeautifulSoup(item.get_body_content(), 'html.parser')
        paragraphs = [p.get_text(strip=True) for p in soup.find_all(['p', 'div'])]
        paragraphs = [p for p in paragraphs if len(p) > 0]
        
        if paragraphs:
            chapters.append(paragraphs)
            
    return chapters

print("parsing EPUB ...")
ja_chapters = extract_paragraphs_from_epub(f'/home/zelin/code/lightnovel_trans/data/j{vol}.epub')
zh_chapters = extract_paragraphs_from_epub(f'/home/zelin/code/lightnovel_trans/data/c{vol}.epub')

print(f"{len(ja_chapters)} -> {len(zh_chapters)}")

In [ ]:
ja_chapters[0]

In [ ]:
ja_chapters_cuthead = ja_chapters[1:-1]
zh_chapters_cuttail = zh_chapters[:-2]
ja_head = ja_chapters[0]

In [ ]:
ja_head = ja_head[2:]

In [ ]:
short_ja_head = [c.replace('\u3000', '') for c in ja_head]
short_ja_head

In [ ]:
ja_head.extend(short_ja_head)
ja_head = set(ja_head)
ja_head

In [ ]:
ja_chapters_concat = [i for c in ja_chapters_cuthead for i in c]
ja_chapters_true = []
prev_i = 1
for i in range(2, len(ja_chapters_concat)):
    c = ja_chapters_concat[i]
    match_head = next((h for h in ja_head if c.startswith(h)), None)
    if match_head:
        ja_chapters_true.append(ja_chapters_concat[prev_i: i])
        prev_i = i + 1
ja_chapters_true.append(ja_chapters_concat[prev_i:])

In [ ]:
ja_chapters_true[4]

In [ ]:
ja_chapters_true[-1][-5:]

In [ ]:
zh_chapters_cuttail[-1][-5:]

## Sentence Alignment

### DTW Method

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

def align_paragraphs_traditional(ja_paras, zh_paras, gap_penalty=0.2):
    if not ja_paras or not zh_paras:
        return []

    ja_embs = embedder.encode(ja_paras, show_progress_bar=False)
    zh_embs = embedder.encode(zh_paras, show_progress_bar=False)

    sim_matrix = cosine_similarity(ja_embs, zh_embs)

    n, m = len(ja_paras), len(zh_paras)
    
    dp = np.zeros((n + 1, m + 1))
    pointers = np.zeros((n + 1, m + 1, 2), dtype=int)

    for i in range(1, n + 1):
        dp[i][0] = dp[i-1][0] - gap_penalty
        pointers[i][0] = [i-1, 0]
    for j in range(1, m + 1):
        dp[0][j] = dp[0][j-1] - gap_penalty
        pointers[0][j] = [0, j-1]

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match_score = sim_matrix[i-1][j-1]
            
            opt_match = dp[i-1][j-1] + match_score
            opt_skip_zh = dp[i-1][j] - gap_penalty
            opt_skip_ja = dp[i][j-1] - gap_penalty

            best_score = max(opt_match, opt_skip_zh, opt_skip_ja)
            dp[i][j] = best_score
            
            if best_score == opt_match:
                pointers[i][j] = [i-1, j-1]
            elif best_score == opt_skip_zh:
                pointers[i][j] = [i-1, j]
            else:
                pointers[i][j] = [i, j-1]

    path = []
    i, j = n, m
    while i > 0 or j > 0:
        prev_i, prev_j = pointers[i][j]
        if prev_i == i - 1 and prev_j == j - 1:
            path.append((prev_i, prev_j)) # 找到一对匹配
        i, j = prev_i, prev_j
        
    path.reverse()

    aligned_data = []
    current_ja_ids = []
    current_zh_ids = []
    
    for (ja_id, zh_id) in path:
        current_ja_ids.append(ja_id)
        current_zh_ids.append(zh_id)

    mapping = {}
    for ja_id, zh_id in path:
        if ja_id not in mapping:
            mapping[ja_id] = []
        mapping[ja_id].append(zh_id)

    final_pairs = []
    for ja_id in sorted(mapping.keys()):
        zh_ids = sorted(list(set(mapping[ja_id])))
        ja_str = ja_paras[ja_id]
        zh_str = "".join([zh_paras[z] for z in zh_ids])
        
        if ja_str.strip() and zh_str.strip():
            final_pairs.append({"ja": ja_str, "zh": zh_str})

    return final_pairs


test_chapter_idx = 1
ja_test = ja_chapters_true[test_chapter_idx]
zh_test = zh_chapters_cuttail[test_chapter_idx]  # 假设这是你的中文变量名

aligned_results = align_paragraphs_traditional(ja_test, zh_test)

for res in aligned_results[:3]:
    print(f"JA: {res['ja']}")
    print(f"ZH: {res['zh']}\n")

In [ ]:
import json
import os
from tqdm.auto import tqdm 

def process_and_save_full_book_dtw(ja_chapters, zh_chapters, output_jsonl="aligned_book_dtw.jsonl"):
    processed_chapters = set()
    if os.path.exists(output_jsonl):
        with open(output_jsonl, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    data = json.loads(line)
                    processed_chapters.add(data["chapter_idx"])
                except json.JSONDecodeError:
                    continue

    total_chapters = min(len(ja_chapters), len(zh_chapters))

    with open(output_jsonl, "a", encoding="utf-8") as f:
        for chapter_idx in tqdm(range(total_chapters)):
            if chapter_idx in processed_chapters:
                continue
                
            ja_paras = ja_chapters[chapter_idx]
            zh_paras = zh_chapters[chapter_idx]
            
            if not ja_paras or not zh_paras:
                continue
            
            aligned_pairs = align_paragraphs_traditional(ja_paras, zh_paras)
            
            chapter_data = {
                "chapter_idx": chapter_idx,
                "pairs": aligned_pairs
            }
            f.write(json.dumps(chapter_data, ensure_ascii=False) + "\n")
            f.flush() 
            
process_and_save_full_book_dtw(ja_chapters_true, zh_chapters_cuttail, output_jsonl=f"aligned_book_dtw_{vol}.jsonl")

In [ ]:
await process_chapter(1, ja_paras=ja_chapters_true[1], zh_paras=zh_chapters_cuttail[1])

In [ ]:
import json
import os

async def process_full_book(ja_chapters, zh_chapters, output_jsonl="aligned_full_book.jsonl"):
    processed_chapters = set()
    if os.path.exists(output_jsonl):
        with open(output_jsonl, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                processed_chapters.add(data["chapter_idx"])
        print(f"{len(processed_chapters)} skipped chapters detected from existing progress.")

    total_chapters = min(len(ja_chapters), len(zh_chapters))
    print(f"{total_chapters} chapters to process...")

    with open(output_jsonl, "a", encoding="utf-8") as f:
        for chapter_idx in range(total_chapters):
            if chapter_idx in processed_chapters:
                continue
                
            print(f"\n--- {chapter_idx+1}/{total_chapters}  ---")
            ja_paras = ja_chapters[chapter_idx]
            zh_paras = zh_chapters[chapter_idx]
            
            aligned_pairs = await process_chapter(chapter_idx, ja_paras, zh_paras)
            
            chapter_data = {
                "chapter_idx": chapter_idx,
                "pairs": aligned_pairs
            }
            f.write(json.dumps(chapter_data, ensure_ascii=False) + "\n")
            f.flush()
            
            print(f"{chapter_idx+1} {len(aligned_pairs)} pairs aligned.")

## Rejected Sample Generation

In [ ]:
import asyncio
import json
import re
import asyncio
from openai import AsyncOpenAI
import pandas as pd
from datasets import Dataset
import re
from tqdm.auto import tqdm

client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)
MODEL_NAME = "Qwen/Qwen3-30B-A3B"
CONCURRENCY_LIMIT = 100 
INPUT_JSONL = f"aligned_book_dtw_{vol}.jsonl"
OUTPUT_PARQUET = f"dpo_dataset_{vol}.parquet"

async def fetch_machine_translation(sem, idx, ja_text):
    async with sem:
        system_instruction = "You are a helpful translation assistant."
        user_prompt = f"Please translate the following Japanese text into Simplified Chinese. Only output the translation, no explanations:\n\n{ja_text}"

        try:
            response = await client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.3,
                max_tokens=1024
            )
            
            content = response.choices[0].message.content
            
            if "<think>" in content:
                content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL)
                
            return idx, content.strip()
            
        except Exception as e:
            print(f"[ID: {idx}]: {e}")
            return idx, ""

async def generate_rejected_and_pack():
    raw_pairs = []
    
    with open(INPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            chapter_data = json.loads(line)
            for pair in chapter_data.get("pairs", []):
                ja = pair.get("ja", "").strip()
                zh = pair.get("zh", "").strip()
                if ja and zh:
                    raw_pairs.append({"ja": ja, "zh": zh})
                    
    total_pairs = len(raw_pairs)

    sem = asyncio.Semaphore(CONCURRENCY_LIMIT)
    tasks = []
    for i, pair in enumerate(raw_pairs):
        tasks.append(fetch_machine_translation(sem, i, pair["ja"]))

    results = await asyncio.gather(*tasks)
    
    rejected_dict = {idx: text for idx, text in results}

    verl_rows = []
    
    training_system_prompt = "你是一位精通中日文学的资深轻小说翻译家。请将以下日文翻译为符合轻小说文风的简体中文，保留角色的口癖和特定文风。"

    for i, pair in tqdm(enumerate(raw_pairs), total=len(raw_pairs), desc="组装 DPO 数据集"):
        rejected_text = rejected_dict.get(i, "")
        
        if rejected_text == "" or rejected_text == pair["zh"]:
            continue
            
        prompt_messages = [
            {"role": "system", "content": training_system_prompt},
            {"role": "user", "content": pair["ja"]}
        ]
        chosen_messages = [
            {"role": "assistant", "content": pair["zh"]}
        ]
        rejected_messages = [
            {"role": "assistant", "content": rejected_text}
        ]
        
        verl_rows.append({
            "prompt": prompt_messages,
            "chosen": chosen_messages,
            "rejected": rejected_messages
        })

    df = pd.DataFrame(verl_rows)
    dataset = Dataset.from_pandas(df)
    dataset.to_parquet(OUTPUT_PARQUET)

In [ ]:
await generate_rejected_and_pack()

In [ ]:
df = pd.read_parquet(OUTPUT_PARQUET)
df

In [ ]:
df.iloc[-1]

In [ ]:
df.iloc[-1]['prompt']

In [ ]:
file_paths = [
    "dpo_dataset_1.parquet", 
    "dpo_dataset_2.parquet", 
    "dpo_dataset_3.parquet"
]

dfs = []
for path in file_paths:
    df = pd.read_parquet(path)
    dfs.append(df)

df_combined = pd.concat(dfs, ignore_index=True)
dataset = Dataset.from_pandas(df_combined)

dataset = dataset.shuffle(seed=42)

split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
HF_USERNAME = "" 
REPO_NAME = ""
repo_id = f"{HF_USERNAME}/{REPO_NAME}"

split_dataset.push_to_hub(repo_id, private=True) 
